<a href="https://colab.research.google.com/github/ncrowder/maven/blob/main/maven_drill_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# data located here: https://mavenanalytics.io/data-drills/the-price-is-right
price = pd.read_csv('price_history.csv')
prod = pd.read_csv('products.csv')
tran = pd.read_csv('transactions.csv')

In [ ]:
prod

,pizza_id,name,current_price
0,1,BBQ Chicken,20.75
1,2,Cheese,18.50
2,3,Chicken Alfredo,20.75
3,4,Hawaiian,20.50
4,5,Margherita,20.75
5,6,Meat Lovers,20.50
6,7,Pepperoni,20.50
7,8,Spicy Italian,20.75
8,9,Supreme,20.75
9,10,Veggie,21.00


In [ ]:
price

,pizza_id,effective_date,price
0,1,2026-01-01,20.75
1,2,2026-01-01,17.95
2,3,2026-01-01,20.75
3,4,2026-01-01,16.50
4,5,2026-01-01,20.25
5,6,2026-01-01,17.50
6,7,2026-01-01,15.25
7,8,2026-01-01,20.25
8,9,2026-01-01,20.75
9,10,2026-01-01,20.25


In [ ]:
price.sort_values(['pizza_id','effective_date']).drop_duplicates('pizza_id',keep='last')

,pizza_id,effective_date,price
0,1,2026-01-01,20.75
12,2,2026-02-02,18.50
2,3,2026-01-01,20.75
19,4,2026-02-16,20.50
14,5,2026-02-02,20.75
20,6,2026-02-16,20.50
21,7,2026-02-16,20.50
17,8,2026-02-02,20.75
8,9,2026-01-01,20.75
18,10,2026-02-02,21.00


In [ ]:
price.loc[price.groupby('pizza_id')['effective_date'].idxmax()]

,pizza_id,effective_date,price
0,1,2026-01-01,20.75
12,2,2026-02-02,18.50
2,3,2026-01-01,20.75
19,4,2026-02-16,20.50
14,5,2026-02-02,20.75
20,6,2026-02-16,20.50
21,7,2026-02-16,20.50
17,8,2026-02-02,20.75
8,9,2026-01-01,20.75
18,10,2026-02-02,21.00


In [ ]:
prod

,pizza_id,name,current_price
0,1,BBQ Chicken,20.75
1,2,Cheese,18.50
2,3,Chicken Alfredo,20.75
3,4,Hawaiian,20.50
4,5,Margherita,20.75
5,6,Meat Lovers,20.50
6,7,Pepperoni,20.50
7,8,Spicy Italian,20.75
8,9,Supreme,20.75
9,10,Veggie,21.00


In [ ]:
tran

,order_detail_id,order_id,order_date,pizza_id,quantity
0,1,1,2026-01-01,4,1
1,2,2,2026-01-01,6,1
2,3,2,2026-01-01,2,1
3,4,2,2026-01-01,9,1
4,5,2,2026-01-01,10,1
...,...,...,...,...,...
7159,7160,3528,2026-02-28,8,1
7160,7161,3529,2026-02-28,7,1
7161,7162,3529,2026-02-28,8,1
7162,7163,3530,2026-02-28,6,1


In [ ]:
def lookup(row):
    df = price[(price.pizza_id == row.pizza_id) & (price.effective_date <= row.order_date)]
    if df.empty:
        return prod[prod.pizza_id == row.pizza_id]['current_price'].iloc[0]
    else:
        return df.tail(1)['price'].iloc[0]

In [ ]:
tran['unit_price'] = tran.apply(lookup,axis=1)

In [ ]:
tran

,order_detail_id,order_id,order_date,pizza_id,quantity,unit_price
0,1,1,2026-01-01,4,1,16.50
1,2,2,2026-01-01,6,1,17.50
2,3,2,2026-01-01,2,1,17.95
3,4,2,2026-01-01,9,1,20.75
4,5,2,2026-01-01,10,1,20.25
...,...,...,...,...,...,...
7159,7160,3528,2026-02-28,8,1,20.75
7160,7161,3529,2026-02-28,7,1,20.50
7161,7162,3529,2026-02-28,8,1,20.75
7162,7163,3530,2026-02-28,6,1,20.50


In [ ]:
tran['price'] = tran['quantity'] * tran['unit_price']

In [ ]:
tran.price.sum()

np.float64(161144.34999999998)

## ChatGPT's Version

In [ ]:
tran = tran.sort_values(['pizza_id', 'order_date'])
price = price.sort_values(['pizza_id', 'effective_date'])

out = pd.merge_asof(
    tran,
    price,
    left_on='order_date',
    right_on='effective_date',
    by='pizza_id',
    direction='backward'
)

out = out.merge(prod[['pizza_id', 'current_price']], on='pizza_id', how='left')

out['final_price'] = out['price'].fillna(out['current_price'])